In [1]:
import scanpy as sc
import os
import anndata
import mudata
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact, hypergeom, pearsonr, spearmanr
import seaborn as sns
import glob
import PyComplexHeatmap as pch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import warnings
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import SpectralCoclustering, SpectralBiclustering
from statsmodels.stats.multitest import multipletests
from pyjaspar import jaspardb

# Read the meta data 

In [2]:
if os.path.exists("/data2st2/junyi/output/stg1028/combined_ALL_meta_match.csv"):
    df_meta_match = pd.read_csv("/data2st2/junyi/output/stg1028/combined_ALL_meta_match.csv", index_col=0)
else:

    df_meta= pd.read_csv("/data2st2/junyi/output/stg1028/combined_ALL_meta.csv", index_col=0)
    df_meta.drop_duplicates(inplace=True)
    df_meta['Region Subclass'] = df_meta['region'] + "_" + df_meta['celltype.L2']
    df_meta_match = df_meta.drop_duplicates(subset=['Region Subclass'])
    df_meta_match['ctname'] = df_meta_match['Region Subclass'].str.replace('/', '-').str.replace(' ', '_')
    df_meta_match.to_csv("/data2st2/junyi/output/stg1028/combined_ALL_meta_match.csv")

In [3]:
def gen_map_from_meta(df_meta_match=df_meta_match,iput_col='ctname',output_col='Region Subclass'):
    df_meta_match = df_meta_match.dropna(subset=[iput_col, output_col])
    df_meta_match = df_meta_match.drop_duplicates(subset=[iput_col, output_col])
    map_dict = dict(zip(df_meta_match[iput_col], df_meta_match[output_col]))
    return map_dict

In [4]:
df_deg_all = pd.read_excel("/data2st2/junyi/output/six_dataset_3v3final_merged_mast_ng_wilcox_degs_fdr_log2fc0.1_filtered_7regions_rmMB_annotation.xlsx")
df_deg_CU3RM = df_deg_all[df_deg_all['Model'].str.contains('CUSUS M')].copy()
df_deg_CU3RM = df_deg_CU3RM[df_deg_CU3RM['Region'].isin(['AMY','HPF','PFC'])].copy()
df_deg_CU3RM['ctname'] = df_deg_CU3RM['Region'] + "_" + df_deg_CU3RM['Subclass']
df_deg_CU3RM['ctname'] = df_deg_CU3RM['ctname'].str.replace('/', '-')
df_deg_CU3RM['ctname'] = df_deg_CU3RM['ctname'].str.replace(' ', '_')
df_deg_CU3RM['targetgene'] = df_deg_CU3RM.Gene

In [5]:
# This is the matrix of tobias score
tobias_all = pd.read_csv(f'/data2st1/junyi/output/atac1112/tobias/tobias_matrix/Jaspar26/Jaspar26_tobias_full_results.csv')

In [6]:
tobias_all['TF_up'] = tobias_all['name'].str.upper()
tobias_all['TF_up'] = tobias_all['TF_up'].str.split('::')
tobias_all_exp = tobias_all.explode('TF_up').reset_index(drop=True)
tobias_all_exp['TF_up'] = tobias_all_exp['TF_up'].str.strip()

In [7]:
TF_jaspr = tobias_all_exp.TF_up.unique().tolist()

In [8]:
comparison_to_status = {
    'CSDS_M': 'CSSUS_M',
    'RES_M': 'CURES_M',
    'RES_F': 'CURES_F',
    'CSDS_F': 'CSSUS_F',
    'SUS_M': 'CUSUS_M',
    'SUS_F': 'CUSUS_F',
    'CSRES_M': 'CSRES_M',
}

## Summarize TF target and DEG overlap

This cell builds a TF-by-comparison summary table that links regulon targets with differential expression results.

- Detects the gene-name columns in `df_s_regulon` and `df_deg_all` and standardizes gene symbols to uppercase.
- Creates a cleaned DEG table with unique target genes per `ctname`, `comparison`, and `Direction`.
- Aggregates total DEG counts, including separate counts for up- and down-regulated genes.
- Reshapes the regulon table into a long format with one TF-target gene pair per row while retaining metadata such as `log2FC`, `FDR`, `sex`, `region`, and `Neurotransmitter_celltype`.
- Merges regulon targets with DEG targets to identify overlaps for each TF within each cell type and comparison.
- Summarizes the overlap into comma-separated target lists and numeric counts, then joins DEG totals for context.

The final output, `df_tf_target_deg_overlap_summary`, reports the TF target gene set, overlapping up/down DEG lists, and the corresponding overlap counts for each `ctname`, `TF`, and `comparison`.

In [9]:
df_tf_target_deg_hypergeom = pd.read_csv('/data2/junyi/stg0901/scenic_wilcoxon_all/scenic_wilcoxon_all_TF_regulon_deg_overlap_summary.csv')

In [10]:
# ---- 读取预计算好的 TF 同源转换映射表 ----
# 转换函数已保存至 convert_tf_mouse_ortholog.py
# 如需重新计算, 运行: python convert_tf_mouse_ortholog.py --tf-list <TF1,TF2,...> --output <path>

# 读取 SCENIC TF 映射
df_scenic_map = pd.read_csv('/data2st1/junyi/output/atac1112/scenic_wilcoxon_all_TF_mapping.csv')
tf_mapping = dict(zip(df_scenic_map['TF_original'], df_scenic_map['TF_official']))

# 读取已保存的完整结果 (含 TF_official, TF_source)
df_tf_target_deg_hypergeom = pd.read_csv(
    '/data2st1/junyi/output/atac1112/scenic_wilcoxon_all_TF_regulon_deg_overlap_summary_with_ortholog.csv'
)
df_tf_target_deg_hypergeom = pd.read_csv('/data2/junyi/stg0901/scenic_wilcoxon_all/scenic_wilcoxon_all_TF_regulon_deg_overlap_summary.csv')

# 读取 TOBIAS (JASPAR) TF 映射
df_tf_jaspr_map = pd.read_csv('/data2st1/junyi/output/atac1112/scenic_wilcoxon_all_TF_jaspr_mapping.csv')
tf_jaspr_mapping = dict(zip(df_tf_jaspr_map['TF_original'], df_tf_jaspr_map['TF_official']))

print(f"SCENIC TF 映射: {len(tf_mapping)} 个")
print(f"TOBIAS TF 映射: {len(tf_jaspr_mapping)} 个")
print(f"df_tf_target_deg_hypergeom 已加载, 形状: {df_tf_target_deg_hypergeom.shape}")
print(f"列: {df_tf_target_deg_hypergeom.columns.tolist()}")

SCENIC TF 映射: 991 个
TOBIAS TF 映射: 887 个
df_tf_target_deg_hypergeom 已加载, 形状: (34172, 20)
列: ['ctname', 'TF', 'comparison', 'log2FC', 'FDR', 'sex', 'region', 'Neurotransmitter_celltype', 'tf_target_geneset', 'n_target_gene', 'n_total_deg', 'n_total_up_deg', 'n_total_down_deg', 'up_deg_list', 'n_up_overlap', 'down_deg_list', 'n_down_overlap', 'hypergeom_p_up', 'hypergeom_p_down', 'celltype.L1']


In [11]:
# ---- 合并 up+down DEG 算 target 基因富集的超几何检验 ----
# 超几何检验参数:
#   population: 全基因组基因数 (老鼠: ~30804)
#   successes_in_pop: n_target_gene (regulon 中的基因数)
#   sample_size: n_total_deg (该 comparison 的总 DEG 数)
#   successes_in_sample: n_up_overlap + n_down_overlap (target 与 DEG 的交集)
if 'hypergeom_p_total' not in df_tf_target_deg_hypergeom.columns:
    n_universe = 30804  # 老鼠基因组基因总数

    df = df_tf_target_deg_hypergeom.copy()

    # 计算合并的 overlap
    df['n_overlap_total'] = df['n_up_overlap'].fillna(0).astype(int) + df['n_down_overlap'].fillna(0).astype(int)

    # 过滤: n_total_deg 不为 NaN, n_target_gene > 0, n_total_deg > 0
    mask = df['n_total_deg'].notna() & (df['n_target_gene'] > 0) & (df['n_total_deg'] > 0)
    df_valid = df[mask].copy()

    # 计算超几何 p 值: P(X >= n_overlap_total)
    df_valid['hypergeom_p_total'] = hypergeom.sf(
        df_valid['n_overlap_total'].values - 1,
        n_universe,
        df_valid['n_target_gene'].values,
        df_valid['n_total_deg'].values.astype(int)
    )

    # 合并回原表
    df_tf_target_deg_hypergeom = df.merge(
        df_valid[['ctname', 'TF', 'comparison', 'hypergeom_p_total']],
        on=['ctname', 'TF', 'comparison'],
        how='left'
    )
    # 补上 n_overlap_total 和 NaN 的 p 值
    df_tf_target_deg_hypergeom['n_overlap_total'] = df['n_overlap_total']
    df_tf_target_deg_hypergeom['hypergeom_p_total'] = df_tf_target_deg_hypergeom['hypergeom_p_total'].fillna(1.0)

    n_valid = df_valid['hypergeom_p_total'].notna().sum()
    print(f"计算完成, 共 {n_valid} 行有有效 p 值")

    print("\n显著富集 (hypergeom_p_total < 0.05) 的 top10:")
    df_sig = df_tf_target_deg_hypergeom[
        df_tf_target_deg_hypergeom['hypergeom_p_total'] < 0.05
    ].sort_values('hypergeom_p_total')
    print(df_sig[['ctname', 'TF', 'comparison', 'n_target_gene', 'n_total_deg', 'n_overlap_total', 'hypergeom_p_total']].head(10))

    # 保存结果
    df_tf_target_deg_hypergeom.to_csv(
        '/data2/junyi/stg0901/merged_result/scenic_wilcoxon_all_TF_regulon_deg_overlap_summary_with_ortholog.csv',
        index=False
    )
    print("\n结果已保存")

计算完成, 共 31748 行有有效 p 值

显著富集 (hypergeom_p_total < 0.05) 的 top10:
                         ctname     TF comparison  n_target_gene  n_total_deg  \
21098      PFC_PFC_L4-5_IT_Glut  NPDC1    CSRES_M            709       1477.0   
32234   TH_TH_Fam20a_Fbln1_Glut  NPDC1    CURES_F           1343       1557.0   
32867    TH_TH_Kcnk13_Nox4_Glut  NPDC1    CURES_F           1207       1715.0   
26804      STR_STR_D1_Fbn2_GABA  NPDC1    CSRES_M            733       1388.0   
13116      HY_HY_Ebf1_Nrn1_Glut   XBP1    CSSUS_M           1084       1662.0   
14203  HY_HY_Lef1_Slc16a10_GABA  NPDC1    CURES_F           1036       1937.0   
28541  STR_STR_Grik1_Kcnc2_GABA   XBP1    CSRES_M            970       1601.0   
13117      HY_HY_Ebf1_Nrn1_Glut   XBP1    CURES_F           1084       1802.0   
32565  TH_TH_Fgf10_Gm16263_Glut  NPDC1    CURES_F           1335       1851.0   
15211      HY_HY_Nts_Ecel1_GABA   XBP1    CURES_F            819       1931.0   

       n_overlap_total  hypergeom_p_total  

In [12]:
df_tf_target_deg_hypergeom['TF_official'] = df_tf_target_deg_hypergeom['TF'].map(tf_mapping)

In [13]:
df_tf_target_deg_hypergeom['TF_official'] = df_tf_target_deg_hypergeom['TF_official'].replace({'NFE2L1':'NRF1'})

In [14]:
df_tf_target_deg_hypergeom

,ctname,TF,comparison,log2FC,FDR,sex,region,Neurotransmitter_celltype,tf_target_geneset,n_target_gene,...,up_deg_list,n_up_overlap,down_deg_list,n_down_overlap,hypergeom_p_up,hypergeom_p_down,celltype.L1,n_overlap_total,hypergeom_p_total,TF_official
0,AMY_AMY_Cav1_Frmpd1_Glut,ARNT2,CURES_F,0.214859,5.308597e-07,F,AMY,Glutamatergic,"1700025G04RIK,AKAP17B,ARNT2,CEMIP,DLGAP4,FHL2,...",19,...,NaN,0,NaN,0,1.000000,1.000000,Glut,0,1.000000,ARNT2
1,AMY_AMY_Cav1_Frmpd1_Glut,ARNT2,CURES_M,0.143422,1.808912e-02,M,AMY,Glutamatergic,"1700025G04RIK,AKAP17B,ARNT2,CEMIP,DLGAP4,FHL2,...",19,...,NaN,0,NaN,0,1.000000,1.000000,Glut,0,1.000000,ARNT2
2,AMY_AMY_Cav1_Frmpd1_Glut,ATF4,CSRES_M,-1.042059,2.414378e-05,M,AMY,Glutamatergic,"1700094D03RIK,2010204K13RIK,2010308F09RIK,2700...",128,...,LRRTM1,1,"CBLN2,GOLGB1,INF2,MAZ,UTP14A",5,0.999962,0.647382,Glut,6,0.338481,ATF4
3,AMY_AMY_Cav1_Frmpd1_Glut,ATF6B,CSRES_M,-0.946801,2.119831e-12,M,AMY,Glutamatergic,"0610005C13RIK,1700110C19RIK,3830406C13RIK,4632...",216,...,"HSP90AB1,PPIA,SRP14,TUBB5",4,"ACVRL1,ADGRB2,CAMKV,CCDC107,ERGIC1,GOLGA7B,GPR...",20,0.999963,0.000968,Glut,24,0.000002,ATF6B
4,AMY_AMY_Cav1_Frmpd1_Glut,ATF7,CSRES_M,-0.493450,6.138779e-07,M,AMY,Glutamatergic,"AK4,AW549877,BMT2,CHFR,EPB41L5,MAPRE3,MPPED1,M...",16,...,NaN,0,CHFR,1,1.000000,0.504308,Glut,1,0.453704,ATF7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34167,TH_VLMC,SIN3A,CUSUS_M,0.556524,2.430031e-02,M,TH,NN,"1110019D14RIK,1700003L19RIK,2610027K06RIK,4933...",200,...,NaN,0,NaN,0,1.000000,1.000000,Vascular,0,1.000000,SIN3A
34168,TH_VLMC,STAT3,CURES_M,1.379796,4.927868e-02,M,TH,NN,"ALDH3B1,ANKFN1,ASB18,ASPA,ATP6V0A4,ATP7B,ATR,C...",94,...,NaN,0,NaN,0,1.000000,1.000000,Vascular,0,1.000000,STAT3
34169,TH_VLMC,TFCP2L1,CUSUS_M,1.585674,3.861692e-02,M,TH,NN,"1810026B05RIK,5830418P13RIK,ADCY6,BOC,CCDC174,...",32,...,NaN,0,NaN,0,1.000000,1.000000,Vascular,0,1.000000,TFCP2L1
34170,TH_VLMC,TGIF1,CSSUS_M,-1.828554,4.610178e-02,M,TH,NN,"ACAA2,CCDC141,CIB2,CISH,CNTN2,EIF2AK3,EZH1,FBX...",20,...,NaN,0,NaN,0,1.000000,1.000000,Vascular,0,1.000000,TGIF1


In [15]:
df_tf_target_deg_hypergeom_filtered = df_tf_target_deg_hypergeom[df_tf_target_deg_hypergeom['TF_official'].isin(df_tf_jaspr_map.TF_official.unique())].copy()
# df_tf_target_deg_hypergeom_filtered = df_tf_target_deg_hypergeom_filtered[
#     (df_tf_target_deg_hypergeom['hypergeom_p_up'] < 0.05) | (df_tf_target_deg_hypergeom['hypergeom_p_down'] < 0.05)
# ]
df_tf_target_deg_hypergeom_filtered = df_tf_target_deg_hypergeom_filtered[
    (df_tf_target_deg_hypergeom['hypergeom_p_total'] < 0.05)
]
df_deg_info = df_deg_all.copy()
df_deg_info['ctname'] = df_deg_all['Region'] + "_" + df_deg_all['Subclass']
df_deg_info['ctname'] = df_deg_info['ctname'].str.replace('/', '-')
df_deg_info['ctname'] = df_deg_info['ctname'].str.replace(' ', '_')
df_deg_info['TF'] = df_deg_info['Gene']
df_deg_info['comparison'] = df_deg_info['Model'].str.replace(' ', '_')
df_deg_info['TF'] = df_deg_info['TF'].str.upper()
df_deg_info['TF_official'] = df_deg_info['TF'].map(tf_mapping)
df_deg_info['Is_DEG'] = True
df_tf_target_deg_hypergeom_filtered['Direction'] = df_tf_target_deg_hypergeom_filtered['log2FC'].apply(lambda x: 'Up' if x > 0 else ('Down' if x < 0 else 'Ns'))
df_tf_target_deg_hypergeom_filtered = df_tf_target_deg_hypergeom_filtered.merge(
    df_deg_info.loc[:, ['log2FC', 'FDR', 'TF','TF_official', 'comparison', 'ctname','Direction','Is_DEG']],
    on=['ctname', 'comparison','TF_official','Direction'],
    how='left',
    suffixes=('_scenic', '_deg')
)
# Change only the first letter of TF to uppercase, rest is lower case
df_tf_target_deg_hypergeom_filtered['TF_up']= df_tf_target_deg_hypergeom_filtered['TF_official']
df_tf_target_deg_hypergeom_filtered['TF'] = df_tf_target_deg_hypergeom_filtered['TF_official'].str.lower().str.capitalize()

/tmp/ipykernel_2087213/1850874520.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_tf_target_deg_hypergeom_filtered = df_tf_target_deg_hypergeom_filtered[
/tmp/ipykernel_2087213/1850874520.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tf_target_deg_hypergeom_filtered['Direction'] = df_tf_target_deg_hypergeom_filtered['log2FC'].apply(lambda x: 'Up' if x > 0 else ('Down' if x < 0 else 'Ns'))


In [16]:
df_tf_target_deg_hypergeom_filtered

,ctname,TF_scenic,comparison,log2FC_scenic,FDR_scenic,sex,region,Neurotransmitter_celltype,tf_target_geneset,n_target_gene,...,n_overlap_total,hypergeom_p_total,TF_official,Direction,log2FC_deg,FDR_deg,TF_deg,Is_DEG,TF_up,TF
0,AMY_AMY_Cav1_Frmpd1_Glut,CREM,CURES_F,-0.639053,5.816175e-07,F,AMY,Glutamatergic,"1110002L01RIK,1600014C10RIK,1810062O18RIK,2900...",167,...,10,0.014181,CREM,Down,NaN,NaN,NaN,NaN,CREM,Crem
1,AMY_AMY_Cav1_Frmpd1_Glut,EGR1,CSRES_M,-0.213135,1.651145e-07,M,AMY,Glutamatergic,"1110002L01RIK,1700003G18RIK,4930488L21RIK,4930...",116,...,11,0.003931,EGR1,Down,NaN,NaN,NaN,NaN,EGR1,Egr1
2,AMY_AMY_Cav1_Frmpd1_Glut,EGR1,CURES_F,0.092994,1.122515e-02,F,AMY,Glutamatergic,"1110002L01RIK,1700003G18RIK,4930488L21RIK,4930...",116,...,13,0.000013,EGR1,Up,NaN,NaN,NaN,NaN,EGR1,Egr1
3,AMY_AMY_Cav1_Frmpd1_Glut,ESRRA,CSRES_M,-1.278235,3.929004e-33,M,AMY,Glutamatergic,"5430405H02RIK,6430548M08RIK,ACOT7,ADCY6,BMP6,C...",77,...,7,0.024014,ESRRA,Down,-0.170954,0.000336,ESRRA,True,ESRRA,Esrra
4,AMY_AMY_Cav1_Frmpd1_Glut,ESRRA,CURES_F,0.396919,1.119416e-06,F,AMY,Glutamatergic,"5430405H02RIK,6430548M08RIK,ACOT7,ADCY6,BMP6,C...",77,...,8,0.001008,ESRRA,Up,0.133270,0.047240,ESRRA,True,ESRRA,Esrra
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8222,TH_TH_Parm1_Mdga1_Glut,ZFP148,CSRES_M,-0.302248,1.885550e-09,M,TH,Glutamatergic,"1500004A13RIK,A330074K22RIK,AATF,BBX,CREBBP,DD...",33,...,3,0.022584,ZFP148,Down,NaN,NaN,NaN,NaN,ZFP148,Zfp148
8223,TH_TH_Parm1_Mdga1_Glut,ZFP362,CSRES_M,0.368821,6.875522e-04,M,TH,Glutamatergic,"B3GAT1,BCL11B,CCDC73,COL12A1,DISP1,EMP3,KIF1B,...",17,...,3,0.003497,ZFP362,Up,NaN,NaN,NaN,NaN,ZFP362,Zfp362
8224,TH_TH_Parm1_Mdga1_Glut,ZFP362,CURES_F,0.456079,7.051668e-03,F,TH,Glutamatergic,"B3GAT1,BCL11B,CCDC73,COL12A1,DISP1,EMP3,KIF1B,...",17,...,3,0.047955,ZFP362,Up,0.110014,0.000007,ZFP362,True,ZFP362,Zfp362
8225,TH_TH_Parm1_Mdga1_Glut,ZFP362,CURES_M,0.385336,1.044262e-02,M,TH,Glutamatergic,"B3GAT1,BCL11B,CCDC73,COL12A1,DISP1,EMP3,KIF1B,...",17,...,4,0.000115,ZFP362,Up,NaN,NaN,NaN,NaN,ZFP362,Zfp362


In [17]:
df_tf_target_deg_hypergeom_filtered['nlogp_scenic'] = -1 * np.log10(df_tf_target_deg_hypergeom_filtered['FDR_scenic'] + 1e-300)
df_tf_target_deg_hypergeom_filtered['nlogp_deg'] = -1 * np.log10(df_tf_target_deg_hypergeom_filtered['FDR_deg'] + 1e-300)


In [18]:
# Generate a named normalized table
df_tf_target_deg_hypergeom_filtered.to_csv('/data2/junyi/stg0901/merged_result/scenic_wilcoxon_all_TF_regulon_deg_overlap_summary_filtered.csv', index=False)

# Process chrom var and tobias

In [19]:
df_Dchrom_all = pd.read_csv("/data2st1/junyi/output/atac1112/chromvar/df_Dchrom_all.csv")


In [20]:
df_Dchrom_all

,names,scores,logfoldchanges,pvals,pvals_adj,celltype.L2,region_nt
0,MA0660.1.MEF2B,12.852792,2.794889,8.296136e-38,8.453763e-35,AMY_Zbtb7c_Vwa5b1_Glut,AMY_AMY_Glut
1,MA0773.1.MEF2D,12.307503,2.563696,8.253768e-35,1.682118e-32,AMY_Zbtb7c_Vwa5b1_Glut,AMY_AMY_Glut
2,MA0497.2.MEF2C,12.020236,2.406506,2.781773e-33,3.149585e-31,AMY_Zbtb7c_Vwa5b1_Glut,AMY_AMY_Glut
3,MA0052.5.MEF2A,11.352599,2.271233,7.199070e-30,3.667926e-28,AMY_Zbtb7c_Vwa5b1_Glut,AMY_AMY_Glut
4,MA2531.1.ZNF775,7.759791,0.770309,8.506922e-15,1.375961e-13,AMY_Zbtb7c_Vwa5b1_Glut,AMY_AMY_Glut
...,...,...,...,...,...,...,...
88648,MA0611.3.Dux,-2.497669,-0.732166,1.250128e-02,7.859037e-01,Ependymal_cell,HIP_NN
88649,MA0846.2.FOXC2,-2.582336,-1.534792,9.813401e-03,7.859037e-01,Ependymal_cell,HIP_NN
88650,MA1524.3.Msgn1,-2.582336,-1.467153,9.813401e-03,7.859037e-01,Ependymal_cell,HIP_NN
88651,MA2586.1.ZNF518B,-2.624669,-1.155498,8.673316e-03,7.859037e-01,Ependymal_cell,HIP_NN


In [21]:
# 对非 NN 的 Neurotransmitter_dar，构建 ctname = Region + "_" + ctname + "_" + Neurotransmitter_dar
df_tf_dar= pd.read_csv('/data2st1/junyi/output/atac1112/dar/celltype.L2/TF_activity_summary.csv')
mask_nn = df_tf_dar['Neurotransmitter_dar'] != 'NN'
df_tf_dar.loc[mask_nn, 'ctname'] = (
    df_tf_dar.loc[mask_nn, 'Region_dar'].astype(str) + '_' +
    df_tf_dar.loc[mask_nn, 'ctname'].astype(str)
)
print(f"更新了 {mask_nn.sum()} 行的 ctname")

更新了 22931 行的 ctname


In [22]:
df_Dchrom_all['Region'] = df_Dchrom_all['region_nt'].str.split('_').str[0]
df_Dchrom_all['Region'] = df_Dchrom_all['Region'].replace('HIP','HPF')

In [23]:
df_Dchrom_all['ctname'] = df_Dchrom_all['Region'] + "_" + df_Dchrom_all['celltype.L2']

In [24]:
df_Dchrom_all['motif'] = df_Dchrom_all['names']
tobias_all['motif'] = tobias_all['motif_id']+"."+tobias_all['name']

In [25]:
tobias_all

,output_prefix,name,motif_id,cluster,total_tfbs,MC_mean_score,MC_bound,MW_mean_score,MW_bound,MC_MW_change,...,Gene,Neurotransmitter,Region Subclass,FDR,Subclass,Region,status,celltype.L2,TF_up,motif
0,Arnt_MA0004.1,Arnt,MA0004.1,C_ARNT::HIF1A,6811,2.30442,790,3.72189,1395,-0.01908,...,Arnt,NN,AMY_Microglia-2,7.696040e-28,Microglia-2,AMY,Down,Microglia-2,[ARNT],MA0004.1.Arnt
1,PAX6_MA0069.1,PAX6,MA0069.1,C_PAX6,4066,1.10674,179,1.85193,318,0.02070,...,PAX6,NN,AMY_Microglia-2,2.684150e-18,Microglia-2,AMY,Up,Microglia-2,[PAX6],MA0069.1.PAX6
2,RORA_MA0071.1,RORA,MA0071.1,C_RORA,8273,0.91384,267,1.87383,673,-0.08059,...,RORA,NN,AMY_Microglia-2,9.984820e-70,Microglia-2,AMY,Down,Microglia-2,[RORA],MA0071.1.RORA
3,RXRAVDR_MA0074.1,RXRA::VDR,MA0074.1,C_RXRA::VDR,5300,1.16380,222,2.10875,489,-0.02211,...,RXRA::VDR,NN,AMY_Microglia-2,5.069410e-23,Microglia-2,AMY,Down,Microglia-2,"[RXRA, VDR]",MA0074.1.RXRA::VDR
4,REL_MA0101.1,REL,MA0101.1,C_RELA,7092,1.41647,449,2.47607,848,-0.02587,...,REL,NN,AMY_Microglia-2,6.857700e-23,Microglia-2,AMY,Down,Microglia-2,[REL],MA0101.1.REL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92724,Nr5a2_MA0505.3,Nr5a2,MA0505.3,C_Esrrg,21889,0.27374,887,0.28166,925,-0.01122,...,Nr5a2,NN,HPF_MOL-1,1.217760e-31,MOL-1,HPF,Down,MOL-1,[NR5A2],MA0505.3.Nr5a2
92725,Nrf1_MA0506.3,Nrf1,MA0506.3,C_Nrf1,18991,1.06643,5867,1.09106,5998,0.03540,...,Nrf1,NN,HPF_MOL-1,7.451180e-72,MOL-1,HPF,Up,MOL-1,[NRF1],MA0506.3.Nrf1
92726,Ptf1A_MA1618.2,Ptf1A,MA1618.2,C_Ptf1A,27675,0.28451,1133,0.29112,1168,0.00470,...,Ptf1A,NN,HPF_MOL-1,8.245600e-10,MOL-1,HPF,Up,MOL-1,[PTF1A],MA1618.2.Ptf1A
92727,Runx1_MA0002.3,Runx1,MA0002.3,C_Bcl11B,28275,0.29644,1385,0.30422,1425,0.00060,...,Runx1,NN,HPF_MOL-1,4.298780e-03,MOL-1,HPF,Up,MOL-1,[RUNX1],MA0002.3.Runx1


In [26]:
jdb_obj = jaspardb(release='JASPAR2026')

In [27]:
tfclass = []
tffamily = []
motif_id = []
for motif in tobias_all['motif_id'].unique():
    try:
        jdb_motif = jdb_obj.fetch_motif_by_id(motif)
        tfclass.append(jdb_motif.tf_class[0] if jdb_motif.tf_class else 'Unknown')
        tffamily.append(jdb_motif.tf_family[0] if jdb_motif.tf_family else 'Unknown')
        motif_id.append(motif)
    except Exception as e:
        print(f"Motif: {motif}, Error: {e}")

In [28]:
df_motif_meta = pd.DataFrame({
    'motif_id': motif_id,
    'tf_class': tfclass,
    'tf_family': tffamily
})

In [29]:
print(tobias_all.columns)
tobias_all

Index(['output_prefix', 'name', 'motif_id', 'cluster', 'total_tfbs',
       'MC_mean_score', 'MC_bound', 'MW_mean_score', 'MW_bound',
       'MC_MW_change', 'MC_MW_pvalue', 'MC_MW_highlighted', 'ctname',
       'source_name', 'abs_change', 'motifmedoid', 'TF', 'Direction', 'log2FC',
       'Sex', 'Gene', 'Neurotransmitter', 'Region Subclass', 'FDR', 'Subclass',
       'Region', 'status', 'celltype.L2', 'TF_up', 'motif'],
      dtype='object')


,output_prefix,name,motif_id,cluster,total_tfbs,MC_mean_score,MC_bound,MW_mean_score,MW_bound,MC_MW_change,...,Gene,Neurotransmitter,Region Subclass,FDR,Subclass,Region,status,celltype.L2,TF_up,motif
0,Arnt_MA0004.1,Arnt,MA0004.1,C_ARNT::HIF1A,6811,2.30442,790,3.72189,1395,-0.01908,...,Arnt,NN,AMY_Microglia-2,7.696040e-28,Microglia-2,AMY,Down,Microglia-2,[ARNT],MA0004.1.Arnt
1,PAX6_MA0069.1,PAX6,MA0069.1,C_PAX6,4066,1.10674,179,1.85193,318,0.02070,...,PAX6,NN,AMY_Microglia-2,2.684150e-18,Microglia-2,AMY,Up,Microglia-2,[PAX6],MA0069.1.PAX6
2,RORA_MA0071.1,RORA,MA0071.1,C_RORA,8273,0.91384,267,1.87383,673,-0.08059,...,RORA,NN,AMY_Microglia-2,9.984820e-70,Microglia-2,AMY,Down,Microglia-2,[RORA],MA0071.1.RORA
3,RXRAVDR_MA0074.1,RXRA::VDR,MA0074.1,C_RXRA::VDR,5300,1.16380,222,2.10875,489,-0.02211,...,RXRA::VDR,NN,AMY_Microglia-2,5.069410e-23,Microglia-2,AMY,Down,Microglia-2,"[RXRA, VDR]",MA0074.1.RXRA::VDR
4,REL_MA0101.1,REL,MA0101.1,C_RELA,7092,1.41647,449,2.47607,848,-0.02587,...,REL,NN,AMY_Microglia-2,6.857700e-23,Microglia-2,AMY,Down,Microglia-2,[REL],MA0101.1.REL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92724,Nr5a2_MA0505.3,Nr5a2,MA0505.3,C_Esrrg,21889,0.27374,887,0.28166,925,-0.01122,...,Nr5a2,NN,HPF_MOL-1,1.217760e-31,MOL-1,HPF,Down,MOL-1,[NR5A2],MA0505.3.Nr5a2
92725,Nrf1_MA0506.3,Nrf1,MA0506.3,C_Nrf1,18991,1.06643,5867,1.09106,5998,0.03540,...,Nrf1,NN,HPF_MOL-1,7.451180e-72,MOL-1,HPF,Up,MOL-1,[NRF1],MA0506.3.Nrf1
92726,Ptf1A_MA1618.2,Ptf1A,MA1618.2,C_Ptf1A,27675,0.28451,1133,0.29112,1168,0.00470,...,Ptf1A,NN,HPF_MOL-1,8.245600e-10,MOL-1,HPF,Up,MOL-1,[PTF1A],MA1618.2.Ptf1A
92727,Runx1_MA0002.3,Runx1,MA0002.3,C_Bcl11B,28275,0.29644,1385,0.30422,1425,0.00060,...,Runx1,NN,HPF_MOL-1,4.298780e-03,MOL-1,HPF,Up,MOL-1,[RUNX1],MA0002.3.Runx1


In [30]:
tobias_all

,output_prefix,name,motif_id,cluster,total_tfbs,MC_mean_score,MC_bound,MW_mean_score,MW_bound,MC_MW_change,...,Gene,Neurotransmitter,Region Subclass,FDR,Subclass,Region,status,celltype.L2,TF_up,motif
0,Arnt_MA0004.1,Arnt,MA0004.1,C_ARNT::HIF1A,6811,2.30442,790,3.72189,1395,-0.01908,...,Arnt,NN,AMY_Microglia-2,7.696040e-28,Microglia-2,AMY,Down,Microglia-2,[ARNT],MA0004.1.Arnt
1,PAX6_MA0069.1,PAX6,MA0069.1,C_PAX6,4066,1.10674,179,1.85193,318,0.02070,...,PAX6,NN,AMY_Microglia-2,2.684150e-18,Microglia-2,AMY,Up,Microglia-2,[PAX6],MA0069.1.PAX6
2,RORA_MA0071.1,RORA,MA0071.1,C_RORA,8273,0.91384,267,1.87383,673,-0.08059,...,RORA,NN,AMY_Microglia-2,9.984820e-70,Microglia-2,AMY,Down,Microglia-2,[RORA],MA0071.1.RORA
3,RXRAVDR_MA0074.1,RXRA::VDR,MA0074.1,C_RXRA::VDR,5300,1.16380,222,2.10875,489,-0.02211,...,RXRA::VDR,NN,AMY_Microglia-2,5.069410e-23,Microglia-2,AMY,Down,Microglia-2,"[RXRA, VDR]",MA0074.1.RXRA::VDR
4,REL_MA0101.1,REL,MA0101.1,C_RELA,7092,1.41647,449,2.47607,848,-0.02587,...,REL,NN,AMY_Microglia-2,6.857700e-23,Microglia-2,AMY,Down,Microglia-2,[REL],MA0101.1.REL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92724,Nr5a2_MA0505.3,Nr5a2,MA0505.3,C_Esrrg,21889,0.27374,887,0.28166,925,-0.01122,...,Nr5a2,NN,HPF_MOL-1,1.217760e-31,MOL-1,HPF,Down,MOL-1,[NR5A2],MA0505.3.Nr5a2
92725,Nrf1_MA0506.3,Nrf1,MA0506.3,C_Nrf1,18991,1.06643,5867,1.09106,5998,0.03540,...,Nrf1,NN,HPF_MOL-1,7.451180e-72,MOL-1,HPF,Up,MOL-1,[NRF1],MA0506.3.Nrf1
92726,Ptf1A_MA1618.2,Ptf1A,MA1618.2,C_Ptf1A,27675,0.28451,1133,0.29112,1168,0.00470,...,Ptf1A,NN,HPF_MOL-1,8.245600e-10,MOL-1,HPF,Up,MOL-1,[PTF1A],MA1618.2.Ptf1A
92727,Runx1_MA0002.3,Runx1,MA0002.3,C_Bcl11B,28275,0.29644,1385,0.30422,1425,0.00060,...,Runx1,NN,HPF_MOL-1,4.298780e-03,MOL-1,HPF,Up,MOL-1,[RUNX1],MA0002.3.Runx1


In [31]:
df_Dchrom_all['Direction'] = df_Dchrom_all['logfoldchanges'].apply(lambda x: 'Up' if x > 0 else ('Down' if x < 0 else 'Ns'))

In [32]:
print(df_Dchrom_all.columns)
df_Dchrom_all

Index(['names', 'scores', 'logfoldchanges', 'pvals', 'pvals_adj',
       'celltype.L2', 'region_nt', 'Region', 'ctname', 'motif', 'Direction'],
      dtype='object')


,names,scores,logfoldchanges,pvals,pvals_adj,celltype.L2,region_nt,Region,ctname,motif,Direction
0,MA0660.1.MEF2B,12.852792,2.794889,8.296136e-38,8.453763e-35,AMY_Zbtb7c_Vwa5b1_Glut,AMY_AMY_Glut,AMY,AMY_AMY_Zbtb7c_Vwa5b1_Glut,MA0660.1.MEF2B,Up
1,MA0773.1.MEF2D,12.307503,2.563696,8.253768e-35,1.682118e-32,AMY_Zbtb7c_Vwa5b1_Glut,AMY_AMY_Glut,AMY,AMY_AMY_Zbtb7c_Vwa5b1_Glut,MA0773.1.MEF2D,Up
2,MA0497.2.MEF2C,12.020236,2.406506,2.781773e-33,3.149585e-31,AMY_Zbtb7c_Vwa5b1_Glut,AMY_AMY_Glut,AMY,AMY_AMY_Zbtb7c_Vwa5b1_Glut,MA0497.2.MEF2C,Up
3,MA0052.5.MEF2A,11.352599,2.271233,7.199070e-30,3.667926e-28,AMY_Zbtb7c_Vwa5b1_Glut,AMY_AMY_Glut,AMY,AMY_AMY_Zbtb7c_Vwa5b1_Glut,MA0052.5.MEF2A,Up
4,MA2531.1.ZNF775,7.759791,0.770309,8.506922e-15,1.375961e-13,AMY_Zbtb7c_Vwa5b1_Glut,AMY_AMY_Glut,AMY,AMY_AMY_Zbtb7c_Vwa5b1_Glut,MA2531.1.ZNF775,Up
...,...,...,...,...,...,...,...,...,...,...,...
88648,MA0611.3.Dux,-2.497669,-0.732166,1.250128e-02,7.859037e-01,Ependymal_cell,HIP_NN,HPF,HPF_Ependymal_cell,MA0611.3.Dux,Down
88649,MA0846.2.FOXC2,-2.582336,-1.534792,9.813401e-03,7.859037e-01,Ependymal_cell,HIP_NN,HPF,HPF_Ependymal_cell,MA0846.2.FOXC2,Down
88650,MA1524.3.Msgn1,-2.582336,-1.467153,9.813401e-03,7.859037e-01,Ependymal_cell,HIP_NN,HPF,HPF_Ependymal_cell,MA1524.3.Msgn1,Down
88651,MA2586.1.ZNF518B,-2.624669,-1.155498,8.673316e-03,7.859037e-01,Ependymal_cell,HIP_NN,HPF,HPF_Ependymal_cell,MA2586.1.ZNF518B,Down


In [33]:
print(df_tf_dar.columns)
df_tf_dar

Index(['ctname', 'motif', 'n_up_genes', 'n_down_genes', 'up_genes',
       'down_genes', 'log2FC_dar_mean', 'n_peaks', 'Neurotransmitter_dar',
       'Region_dar', 'has_DAR'],
      dtype='object')


,ctname,motif,n_up_genes,n_down_genes,up_genes,down_genes,log2FC_dar_mean,n_peaks,Neurotransmitter_dar,Region_dar,has_DAR
0,AMY_Astrocyte-1,MA0003.5.TFAP2A,1,2,Actb,Zhx3,0.012535,2,NN,AMY,True
1,AMY_Astrocyte-1,MA0004.1.Arnt,0,2,NaN,"Fam20a,Snrnp70",-0.073958,2,NN,AMY,True
2,AMY_Astrocyte-1,MA0006.2.Ahr::Arnt,2,2,"Hnrnpdl,Wwox","Fam20a,Snrnp70",0.032082,4,NN,AMY,True
3,AMY_Astrocyte-1,MA0014.4.PAX5,0,1,NaN,Fam20a,-0.056161,1,NN,AMY,True
4,AMY_Astrocyte-1,MA0017.3.NR2F1,2,0,"Agt,Rab34",NaN,0.139648,2,NN,AMY,True
...,...,...,...,...,...,...,...,...,...,...,...
27029,PFC_PFC_PFC_PFC_Sst_GABA,MA2683.1.NPAS3,0,2,NaN,Fgf1,-0.066281,1,GABA,PFC,True
27030,PFC_PFC_PFC_PFC_Sst_GABA,MA2686.1.ZEB2,0,1,NaN,Pde10a,-0.039228,1,GABA,PFC,True
27031,PFC_PFC_PFC_PFC_Sst_GABA,MA2687.1.ZNF251,0,1,NaN,Pde10a,-0.039228,1,GABA,PFC,True
27032,PFC_PFC_PFC_PFC_Sst_GABA,MA2688.1.ZNF347,0,5,NaN,"Elavl2,Fgf1,Pitpnc1,Usp29",-0.056676,4,GABA,PFC,True


In [34]:
df_tobias_all_selected = tobias_all[['motif','ctname','Region','Neurotransmitter','Direction','log2FC','FDR']].copy()
df_chrVAR_all_selected = df_Dchrom_all[['motif','ctname','Region','Direction','logfoldchanges','pvals_adj']].copy()
df_chrVAR_all_selected.rename(columns={'logfoldchanges':'log2FC','pvals_adj':'FDR'}, inplace=True)
df_tfdar_selected = df_tf_dar.copy()

In [35]:
df_tobias_all_selected['has_tobias'] = True
df_chrVAR_all_selected['has_chrVAR'] = True

In [36]:
tobias_all_filtered = tobias_all[tobias_all.FDR < 0.05].copy()
chrVAR_all_filtered = df_Dchrom_all[df_Dchrom_all.pvals_adj < 0.05].copy()

In [37]:
chrVAR_all_filtered.to_csv('/data2st1/junyi/output/atac1112/chromvar/df_Dchrom_all_filtered.csv', index=False)
tobias_all_filtered.to_csv('/data2st1/junyi/output/atac1112/tobias/tobias_all_filtered.csv', index=False)

In [38]:
df_tf_dar.to_csv('/data2st1/junyi/output/atac1112/dar/celltype.L2/TF_activity_summary.csv', index=False)

In [39]:
#join three tables using motif, ctname
df_joined = df_tobias_all_selected.merge(df_chrVAR_all_selected, on=['motif', 'ctname'], how='outer', suffixes=('_tobias', '_chrVAR'))
df_joined = df_joined.merge(df_tfdar_selected, on=['motif', 'ctname'], how='outer', suffixes=('_merged', '_tf_dar'))

In [40]:
df_joined['motif_name'] = df_joined['motif'].str.split('.').str[-1]
df_joined['motif_id'] = df_joined['motif'].str.split('.').str[:-1].str.join('.')

In [41]:
df_joined['VarTob_consist'] = False

In [42]:
df_joined.loc[df_joined['Direction_tobias']== df_joined['Direction_chrVAR'], 'VarTob_consist'] = True

In [43]:
df_joined = df_joined.merge(df_motif_meta, left_on='motif_id', right_on='motif_id', how='left')

In [44]:
df_joined['Sig_chrVAR'] = df_joined['FDR_chrVAR'] < 0.05
df_joined['Sig_tobias'] = df_joined['FDR_tobias'] < 0.05

In [45]:
df_joined.to_csv('/data2/junyi/stg0901/merged_result/combined_TF_table.csv', index=False)

In [46]:
df_joined['PosTF'] = df_joined['motif_name']

In [47]:
# 将 PosTF 中的 :: 拆开，一行变多行（每个 TF 单独一行）
df_joined = df_joined.assign(
    PosTF=lambda x: x['PosTF'].str.split('::')
).explode('PosTF').reset_index(drop=True)
print(f"展开后共 {len(df_joined)} 行")

展开后共 128788 行


In [48]:
# 将 PosTF 转换为老鼠基因名（使用 tf_jaspr_mapping）
df_joined['PosTF_up'] = df_joined['PosTF'].str.upper().str.strip()
df_joined['TF_official'] = df_joined['PosTF_up'].map(tf_jaspr_mapping)
unconverted = df_joined['TF_official'].isna().sum()
df_joined['TF_official'] = df_joined['TF_official'].fillna(df_joined['PosTF_up'])
print(f"转换完成, {len(df_joined)} 行, 未转换: {unconverted} 行")

转换完成, 128788 行, 未转换: 0 行


In [49]:
df_joined

,motif,ctname,Region_tobias,Neurotransmitter,Direction_tobias,log2FC_tobias,FDR_tobias,has_tobias,Region_chrVAR,Direction_chrVAR,...,motif_name,motif_id,VarTob_consist,tf_class,tf_family,Sig_chrVAR,Sig_tobias,PosTF,PosTF_up,TF_official
0,MA0004.1.Arnt,AMY_Microglia-2,AMY,NN,Down,-0.01908,7.696040e-28,True,AMY,Down,...,Arnt,MA0004.1,True,Basic helix-loop-helix factors (bHLH),PAS domain factors,False,True,Arnt,ARNT,ARNT
1,MA0069.1.PAX6,AMY_Microglia-2,AMY,NN,Up,0.02070,2.684150e-18,True,AMY,Up,...,PAX6,MA0069.1,True,Paired box factors,Paired plus homeo domain,False,True,PAX6,PAX6,PAX6
2,MA0071.1.RORA,AMY_Microglia-2,AMY,NN,Down,-0.08059,9.984820e-70,True,AMY,Down,...,RORA,MA0071.1,True,Nuclear receptors with C4 zinc fingers,Thyroid hormone receptor-related factors (NR1),False,True,RORA,RORA,RORA
3,MA0074.1.RXRA::VDR,AMY_Microglia-2,AMY,NN,Down,-0.02211,5.069410e-23,True,AMY,Down,...,RXRA::VDR,MA0074.1,True,Nuclear receptors with C4 zinc fingers,RXR-related receptors (NR2),False,True,RXRA,RXRA,RXRA
4,MA0074.1.RXRA::VDR,AMY_Microglia-2,AMY,NN,Down,-0.02211,5.069410e-23,True,AMY,Down,...,RXRA::VDR,MA0074.1,True,Nuclear receptors with C4 zinc fingers,RXR-related receptors (NR2),False,True,VDR,VDR,VDR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128783,MA2683.1.NPAS3,PFC_PFC_PFC_PFC_Sst_GABA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NPAS3,MA2683.1,False,Basic helix-loop-helix factors (bHLH),PAS,False,False,NPAS3,NPAS3,NPAS3
128784,MA2686.1.ZEB2,PFC_PFC_PFC_PFC_Sst_GABA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ZEB2,MA2686.1,False,Homeo domain factors,HD-ZF,False,False,ZEB2,ZEB2,ZEB2
128785,MA2687.1.ZNF251,PFC_PFC_PFC_PFC_Sst_GABA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ZNF251,MA2687.1,False,C2H2 zinc finger factors,Factors with multiple dispersed zinc fingers,False,False,ZNF251,ZNF251,ZFP251
128786,MA2688.1.ZNF347,PFC_PFC_PFC_PFC_Sst_GABA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ZNF347,MA2688.1,False,C2H2 zinc finger factors,More than 3 adjacent zinc fingers,False,False,ZNF347,ZNF347,ZNF347


In [50]:
df_scenic_M3R = df_tf_target_deg_hypergeom_filtered[df_tf_target_deg_hypergeom_filtered['comparison'].str.contains('CUSUS_M')].copy()
df_scenic_M3R = df_scenic_M3R[df_scenic_M3R.region.isin(['AMY','HPF','PFC'])].copy()
df_all_merged = df_scenic_M3R.merge(
    df_joined,
    left_on=['TF_official','ctname'],
    right_on=['TF_official','ctname'],
    how='left',
    suffixes=('_scenic', '_combined')
)

In [51]:
df_all_merged.columns
df_all_merged

,ctname,TF_scenic,comparison,log2FC_scenic,FDR_scenic,sex,region,Neurotransmitter_celltype,tf_target_geneset,n_target_gene,...,has_DAR,motif_name,motif_id,VarTob_consist,tf_class,tf_family,Sig_chrVAR,Sig_tobias,PosTF,PosTF_up
0,AMY_AMY_Ccdc3_Acvr1c_Glut,EGR4,CUSUS_M,0.262836,1.481991e-02,M,AMY,Glutamatergic,"ACTG1,ADAMTS1,ADRB1,ARC,ARL5B,BC031181,BCL2,BC...",92,...,NaN,EGR4,MA0733.2,False,C2H2 zinc finger factors,Three-zinc finger Kruppel-related,False,True,EGR4,EGR4
1,AMY_AMY_Ccdc3_Acvr1c_Glut,ESRRA,CUSUS_M,-0.244577,8.651608e-07,M,AMY,Glutamatergic,"AARS,ABHD11,ACSL1,ATP6V0A1,CACNB3,CAMKV,CCZ1,C...",43,...,NaN,ESRRA,MA0592.4,True,Nuclear receptors with C4 zinc fingers,Steroid hormone receptors (NR3),False,True,ESRRA,ESRRA
2,AMY_AMY_Ccdc3_Acvr1c_Glut,FOSL2,CUSUS_M,0.035046,2.132040e-02,M,AMY,Glutamatergic,"2610035D17RIK,2610203C22RIK,4933413L06RIK,A330...",181,...,NaN,FOSL2,MA0478.2,True,Basic leucine zipper factors (bZIP),Fos-related,False,True,FOSL2,FOSL2
3,AMY_AMY_Ccdc3_Acvr1c_Glut,FOSL2,CUSUS_M,0.035046,2.132040e-02,M,AMY,Glutamatergic,"2610035D17RIK,2610203C22RIK,4933413L06RIK,A330...",181,...,NaN,FOSL2::JUN,MA1130.2,True,Basic leucine zipper factors (bZIP),Fos-related,False,True,FOSL2,FOSL2
4,AMY_AMY_Ccdc3_Acvr1c_Glut,FOSL2,CUSUS_M,0.035046,2.132040e-02,M,AMY,Glutamatergic,"2610035D17RIK,2610203C22RIK,4933413L06RIK,A330...",181,...,NaN,FOSL2::JUN,MA1131.2,False,Basic leucine zipper factors (bZIP),Fos-related,False,True,FOSL2,FOSL2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882,PFC_PFC_L6_IT_Glut,FOSL2,CUSUS_M,-0.185596,4.449322e-07,M,PFC,Glutamatergic,"1700110K17RIK,2010300C02RIK,4932438A13RIK,4933...",223,...,NaN,FOSL2::JUND,MA1145.2,True,Basic leucine zipper factors (bZIP),Fos-related,False,True,FOSL2,FOSL2
883,PFC_PFC_L6_IT_Glut,SREBF2,CUSUS_M,-0.087217,1.351480e-02,M,PFC,Glutamatergic,"ADGRB2,ADGRL1,ADGRL3,AUTS2,CACNA1G,CALR,CELF4,...",30,...,NaN,SREBF2,MA0596.1,False,Basic helix-loop-helix factors (bHLH),bHLH-ZIP,False,True,SREBF2,SREBF2
884,PFC_PFC_L6_IT_Glut,SREBF2,CUSUS_M,-0.087217,1.351480e-02,M,PFC,Glutamatergic,"ADGRB2,ADGRL1,ADGRL3,AUTS2,CACNA1G,CALR,CELF4,...",30,...,NaN,ELK1::SREBF2,MA1933.2,False,Tryptophan cluster factors,Ets-related,False,True,SREBF2,SREBF2
885,PFC_PFC_L6_IT_Glut,SREBF2,CUSUS_M,-0.087217,1.351480e-02,M,PFC,Glutamatergic,"ADGRB2,ADGRL1,ADGRL3,AUTS2,CACNA1G,CALR,CELF4,...",30,...,NaN,SREBF2,MA0828.3,True,Basic helix-loop-helix factors (bHLH),bHLH-ZIP,True,True,SREBF2,SREBF2


In [52]:
df_all_merged

,ctname,TF_scenic,comparison,log2FC_scenic,FDR_scenic,sex,region,Neurotransmitter_celltype,tf_target_geneset,n_target_gene,...,has_DAR,motif_name,motif_id,VarTob_consist,tf_class,tf_family,Sig_chrVAR,Sig_tobias,PosTF,PosTF_up
0,AMY_AMY_Ccdc3_Acvr1c_Glut,EGR4,CUSUS_M,0.262836,1.481991e-02,M,AMY,Glutamatergic,"ACTG1,ADAMTS1,ADRB1,ARC,ARL5B,BC031181,BCL2,BC...",92,...,NaN,EGR4,MA0733.2,False,C2H2 zinc finger factors,Three-zinc finger Kruppel-related,False,True,EGR4,EGR4
1,AMY_AMY_Ccdc3_Acvr1c_Glut,ESRRA,CUSUS_M,-0.244577,8.651608e-07,M,AMY,Glutamatergic,"AARS,ABHD11,ACSL1,ATP6V0A1,CACNB3,CAMKV,CCZ1,C...",43,...,NaN,ESRRA,MA0592.4,True,Nuclear receptors with C4 zinc fingers,Steroid hormone receptors (NR3),False,True,ESRRA,ESRRA
2,AMY_AMY_Ccdc3_Acvr1c_Glut,FOSL2,CUSUS_M,0.035046,2.132040e-02,M,AMY,Glutamatergic,"2610035D17RIK,2610203C22RIK,4933413L06RIK,A330...",181,...,NaN,FOSL2,MA0478.2,True,Basic leucine zipper factors (bZIP),Fos-related,False,True,FOSL2,FOSL2
3,AMY_AMY_Ccdc3_Acvr1c_Glut,FOSL2,CUSUS_M,0.035046,2.132040e-02,M,AMY,Glutamatergic,"2610035D17RIK,2610203C22RIK,4933413L06RIK,A330...",181,...,NaN,FOSL2::JUN,MA1130.2,True,Basic leucine zipper factors (bZIP),Fos-related,False,True,FOSL2,FOSL2
4,AMY_AMY_Ccdc3_Acvr1c_Glut,FOSL2,CUSUS_M,0.035046,2.132040e-02,M,AMY,Glutamatergic,"2610035D17RIK,2610203C22RIK,4933413L06RIK,A330...",181,...,NaN,FOSL2::JUN,MA1131.2,False,Basic leucine zipper factors (bZIP),Fos-related,False,True,FOSL2,FOSL2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882,PFC_PFC_L6_IT_Glut,FOSL2,CUSUS_M,-0.185596,4.449322e-07,M,PFC,Glutamatergic,"1700110K17RIK,2010300C02RIK,4932438A13RIK,4933...",223,...,NaN,FOSL2::JUND,MA1145.2,True,Basic leucine zipper factors (bZIP),Fos-related,False,True,FOSL2,FOSL2
883,PFC_PFC_L6_IT_Glut,SREBF2,CUSUS_M,-0.087217,1.351480e-02,M,PFC,Glutamatergic,"ADGRB2,ADGRL1,ADGRL3,AUTS2,CACNA1G,CALR,CELF4,...",30,...,NaN,SREBF2,MA0596.1,False,Basic helix-loop-helix factors (bHLH),bHLH-ZIP,False,True,SREBF2,SREBF2
884,PFC_PFC_L6_IT_Glut,SREBF2,CUSUS_M,-0.087217,1.351480e-02,M,PFC,Glutamatergic,"ADGRB2,ADGRL1,ADGRL3,AUTS2,CACNA1G,CALR,CELF4,...",30,...,NaN,ELK1::SREBF2,MA1933.2,False,Tryptophan cluster factors,Ets-related,False,True,SREBF2,SREBF2
885,PFC_PFC_L6_IT_Glut,SREBF2,CUSUS_M,-0.087217,1.351480e-02,M,PFC,Glutamatergic,"ADGRB2,ADGRL1,ADGRL3,AUTS2,CACNA1G,CALR,CELF4,...",30,...,NaN,SREBF2,MA0828.3,True,Basic helix-loop-helix factors (bHLH),bHLH-ZIP,True,True,SREBF2,SREBF2


In [53]:
df_all_merged_result = df_all_merged.drop(columns=['TF_scenic','TF_up','TF','Region_chrVAR','Neurotransmitter_dar','Region_tobias','Region_dar','PosTF','PosTF_up'])
df_all_merged_result = df_all_merged_result.rename(columns={
    'n_up_genes': 'dar_link_n_up_genes',
    'n_down_genes': 'dar_link_n_down_genes',
    'up_genes': 'dar_link_up_genes',
    'down_genes': 'dar_link_down_genes',
})

In [54]:
# 排序列名：细胞类型 + TF 信息排前面
prefix_order = ['ctname','celltype.L1', 'TF_official', 'comparison', 'region', 'Neurotransmitter',
                'Direction', 'log2FC', 'FDR', 'nlogp',
                'n_target', 'n_total_deg', 'n_overlap', 'hypergeom',
                'motif', 'tf_class', 'tf_family',
                'VarTob_consist', 'Sig', 'has_',
                'dar_link_',
                'n_peaks', 'has_DAR']

cols = df_all_merged_result.columns.tolist()

def sort_key(col):
    col_lower = col.lower()
    for i, prefix in enumerate(prefix_order):
        if col_lower.startswith(prefix.lower()):
            return (i, col)
    return (len(prefix_order), col)

ordered_cols = sorted(cols, key=sort_key)
df_all_merged_result = df_all_merged_result[ordered_cols]
print("重排后的列:", ordered_cols)
df_all_merged_result.to_csv('/data2/junyi/stg0901/merged_result/combined_3r_table_final.csv', index=False)

重排后的列: ['ctname', 'celltype.L1', 'TF_official', 'comparison', 'region', 'Neurotransmitter', 'Neurotransmitter_celltype', 'Direction', 'Direction_chrVAR', 'Direction_tobias', 'log2FC_chrVAR', 'log2FC_dar_mean', 'log2FC_deg', 'log2FC_scenic', 'log2FC_tobias', 'FDR_chrVAR', 'FDR_deg', 'FDR_scenic', 'FDR_tobias', 'nlogp_deg', 'nlogp_scenic', 'n_target_gene', 'n_total_deg', 'n_overlap_total', 'hypergeom_p_down', 'hypergeom_p_total', 'hypergeom_p_up', 'motif', 'motif_id', 'motif_name', 'tf_class', 'tf_family', 'VarTob_consist', 'Sig_chrVAR', 'Sig_tobias', 'has_DAR', 'has_chrVAR', 'has_tobias', 'dar_link_down_genes', 'dar_link_n_down_genes', 'dar_link_n_up_genes', 'dar_link_up_genes', 'n_peaks', 'Is_DEG', 'TF_deg', 'down_deg_list', 'n_down_overlap', 'n_total_down_deg', 'n_total_up_deg', 'n_up_overlap', 'sex', 'tf_target_geneset', 'up_deg_list']
